# ViSongSearch: crawl Hợp Âm Chuẩn (chỉ cần CPU + Internet)

Dùng notebook này **chỉ khi** môi trường cloud của project vẫn chặn hopamchuan.com. Crawler đi lịch sự: 1 yêu cầu/giây, tuân theo robots.txt, tự giới thiệu là crawler nghiên cứu của sinh viên.

1. *Add Input* → dataset `visongsearch-bundle`. *Settings*: Accelerator = **None** (không tốn quota GPU), **Internet = On**.
2. **Save Version → Save & Run All** để chạy nền (tối đa 12 giờ; ~15.000 bài cần khoảng 6–9 giờ).
3. Xong thì tải `hac.jsonl` ở tab *Output* và gửi vào thread của project. Nếu bị dừng giữa chừng, chạy lại: crawler tiếp tục từ chỗ cũ khi bạn thêm output cũ (`hac.sqlite`) làm input.

In [ ]:
import glob, os, shutil
ROOT = "/kaggle/working/vss"
if not os.path.exists(ROOT):
    inits = glob.glob("/kaggle/input/**/visong/__init__.py", recursive=True)
    if not inits:
        raise SystemExit("Add the visongsearch-bundle dataset first (Add Input).")
    shutil.copytree(os.path.dirname(os.path.dirname(inits[0])), ROOT)
os.chdir(ROOT)
DB = "/kaggle/working/hac.sqlite"
old = glob.glob("/kaggle/input/**/hac.sqlite", recursive=True)      # resume from an earlier run's output
if old and not os.path.exists(DB):
    shutil.copy(old[0], DB)
    print("resuming from", old[0])

In [ ]:
# Check the parser on three known song pages before the long crawl
!python crawler/hac_crawler.py --test 10416 11844 2322

In [ ]:
# Crawl (stops by itself after 11 hours or 15,000 songs; safe to re-run)
!python crawler/hac_crawler.py --db {DB} --max-songs 15000 --hours 11 2>&1 | tail -20

In [ ]:
!python crawler/hac_crawler.py --db {DB} --export /kaggle/working/hac.jsonl
import sqlite3
con = sqlite3.connect(DB)
print("songs:", con.execute("select count(*) from songs").fetchone()[0],
      "| pages tried:", con.execute("select count(*) from fetched").fetchone()[0],
      "| status codes:", con.execute("select status, count(*) from fetched group by status").fetchall())
print(round(os.path.getsize("/kaggle/working/hac.jsonl") / 1e6, 1), "MB")